# Step 03 — TomTom traffic quick check



In [ ]:
import pickle
import sys

import networkx as nx
import pandas as pd
from IPython.display import display

from routing.config import (
    MAP_RADIUS_M, ORIGIN, DESTINATION, RESULTS_DIR,
    MAX_TRAFFIC_REQUESTS, MIN_TOMTOM_CONFIDENCE, TRAFFIC_REQUEST_PAUSE_S,
)
from routing.map_loader import load_road_graph
from routing.map_view import interactive_map
from routing.traffic_api import TomTomClient
from routing.traffic_pipeline import (
    preview_probes, collect_traffic_snapshot, make_routing_graph,
)

print("Python:", sys.executable)
G = load_road_graph(radius_m=MAP_RADIUS_M)
print("OSM graph:", G.number_of_nodes(), "nodes,", G.number_of_edges(), "directed edges")
display(pd.DataFrame(preview_probes(G, limit=5)))
print("Probe preview does not make API requests.")

Python kernel: /Users/mojtabashabani/wiser/mary/.venv/bin/python
OSM nodes: 747
Directed edges: 1718


In [ ]:
RUN_LIVE_TRAFFIC = False
REQUEST_LIMIT = min(6, MAX_TRAFFIC_REQUESTS)

if RUN_LIVE_TRAFFIC:
    with TomTomClient() as client:
        snapshot, summary = collect_traffic_snapshot(
            G,
            client,
            max_requests=REQUEST_LIMIT,
            pause_s=TRAFFIC_REQUEST_PAUSE_S,
            confidence_threshold=MIN_TOMTOM_CONFIDENCE,
        )
    print("Requests:", summary["requests_made"])
    print("Errors:", summary["request_errors"])
    print("Observed coverage:", summary["usable_coverage_percent"], "%")
    print("Traffic counts:", summary["counts"])
else:
    snapshot_path = RESULTS_DIR / "traffic_snapshot.pkl"
    snapshot = None
    summary = {}
    if snapshot_path.exists():
        with snapshot_path.open("rb") as file:
            bundle = pickle.load(file)
        snapshot = bundle.get("snapshot") if isinstance(bundle, dict) else bundle
        summary = bundle.get("summary", {}) if isinstance(bundle, dict) else {}
        if not isinstance(snapshot, nx.MultiDiGraph):
            raise TypeError("Saved traffic snapshot has an unsupported format")
        if set(snapshot.edges(keys=True)) != set(G.edges(keys=True)):
            raise ValueError("Saved snapshot does not match the configured road graph")
        print("Loaded saved traffic snapshot; no API request was made.")
    else:
        snapshot, summary = collect_traffic_snapshot(
            G, client=None, max_requests=0, pause_s=0
        )
        print("No saved snapshot exists; showing a temporary offline snapshot.")

## Display the road and traffic map

In [ ]:
traffic_widget, state = interactive_map(
    G,
    origin=ORIGIN,
    destination=DESTINATION,
    traffic_graph=snapshot,
)
display(traffic_widget)
print("Observed traffic coverage:", state["traffic_coverage_percent"], "%")
print("Traffic counts:", state["traffic_counts"])

## Prepare routing graphs

In [ ]:
time_graph = make_routing_graph(snapshot, objective="time", scenario="measured")
distance_graph = make_routing_graph(snapshot, objective="distance", scenario="measured")
print("Traffic-weighted time edges:", time_graph.number_of_edges())
print("Distance-weighted edges:", distance_graph.number_of_edges())
print("Missing traffic observations still use imputed fallback costs.")